# CHRONOS

## IMPORTS

In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from chronos import ChronosPipeline
from ucimlrepo import fetch_ucirepo

## DATASET LOADING

In [ ]:
# fetch dataset 
bike_sharing = fetch_ucirepo(id=275) 
  
# original df 
df = bike_sharing.data.original

# variable information 
print(bike_sharing.variables)

print(df.head())

## CHRONOS MODELING

### Data preparation

In [ ]:
df['dteday'] = pd.to_datetime(df['dteday'])

train_size = int(len(df) * 0.80)
train, test = df.iloc[:train_size], df.iloc[train_size:]

y_test_registered = test["registered"]
y_test_casual = test["casual"]

### Chronos Model Initialization

In [ ]:
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map='cuda' if torch.cuda.is_available() else 'cpu',
    torch_dtype=torch.bfloat16,
)

### Walk Forward

In [ ]:
test_length = len(y_test_registered)
step_size = 24
context_length = 512

chronos_preds_reg = []
chronos_preds_cas = []
all_chronos_preds = []

print(f"Starting Walk-Forward with Chronos for {test_length // 24} days...")

history_reg = pd.concat([train['registered'], test['registered']])
history_cas = pd.concat([train['casual'], test['casual']])
test_start_idx = len(train)

# =====================================================================
# 2. BUCLE WALK-FORWARD
# =====================================================================
for i in tqdm(range(0, test_length, step_size), desc="Simulating days"):
    
    current_step = min(step_size, test_length - i)

    current_time_idx = test_start_idx + i
    context_reg = history_reg.iloc[current_time_idx - context_length : current_time_idx].values
    context_cas = history_cas.iloc[current_time_idx - context_length : current_time_idx].values

    tensor_reg = torch.tensor(context_reg, dtype=torch.float32)
    tensor_cas = torch.tensor(context_cas, dtype=torch.float32)
    
    forecast_reg = pipeline.predict(tensor_reg, prediction_length=current_step)
    forecast_cas = pipeline.predict(tensor_cas, prediction_length=current_step)
    
    pred_reg_median = np.quantile(forecast_reg[0].numpy(), 0.5, axis=0)
    pred_cas_median = np.quantile(forecast_cas[0].numpy(), 0.5, axis=0)
    
    chronos_preds_reg.extend(pred_reg_median)
    chronos_preds_cas.extend(pred_cas_median)
    all_chronos_preds.extend(pred_reg_median + pred_cas_median)
# =====================================================================
# 3. EVALUACIÓN FINAL
# =====================================================================
# Convertimos a serie para comparar con el 'cnt' real de test
final_pred_series = pd.Series(all_chronos_preds, index=test[:len(all_chronos_preds)].index)
final_pred_reg = pd.Series(chronos_preds_reg, index=test[:len(chronos_preds_reg)].index)
final_pred_cas = pd.Series(chronos_preds_cas, index=test[:len(chronos_preds_cas)].index)

actual_reg = test['registered'].iloc[:len(all_chronos_preds)]
actual_cas = test['casual'].iloc[:len(all_chronos_preds)]
actual_cnt = test['cnt'].iloc[:len(all_chronos_preds)]

print(f"\n--- FINAL RESULT WALK-FORWARD (140 days) ---")

# 2. Cálculos
from sklearn.metrics import mean_absolute_error

mae_cnt = mean_absolute_error(actual_cnt, final_pred_series)
mae_reg = mean_absolute_error(actual_reg, final_pred_reg)
mae_cas = mean_absolute_error(actual_cas, final_pred_cas)

wmape_cnt = np.sum(np.abs(actual_cnt - final_pred_series)) / np.sum(actual_cnt)
wmape_reg = np.sum(np.abs(actual_reg - final_pred_reg)) / np.sum(actual_reg)
wmape_cas = np.sum(np.abs(actual_cas - final_pred_cas)) / np.sum(actual_cas)

accuracy_cnt = 1 - wmape_cnt
accuracy_reg = 1 - wmape_reg
accuracy_cas = 1 - wmape_cas

# 3. Prints
label = "CHRONOS"
print(f"MAE CNT({label}): {mae_cnt:.2f}")
print(f"MAE REGISTERED({label}): {mae_reg:.2f}")
print(f"MAE CASUAL({label}): {mae_cas:.2f}")
print("-------------------------------")
print(f"WMAPE CNT({label}): {wmape_cnt * 100:.2f}%")
print(f"WMAPE REGISTERED({label}): {wmape_reg * 100:.2f}%")
print(f"WMAPE CASUAL({label}): {wmape_cas * 100:.2f}%")
print("-------------------------------")
print(f"ACCURACY CNT({label}): {accuracy_cnt * 100:.2f}%")
print(f"ACCURACY REGISTERED({label}): {accuracy_reg * 100:.2f}%")
print(f"ACCURACY CASUAL({label}): {accuracy_cas * 100:.2f}%")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train["cnt"].index[-168:], train["cnt"].iloc[-168:], label='Train (Last week)', color='gray')
ax.plot(actual_cnt.index, actual_cnt, label='Test (Reality)', color='blue')
ax.plot(actual_cnt.index, final_pred_series, label=f'{label} (24h)', color='green', linestyle='--')
ax.axvspan(actual_cnt.index[0], actual_cnt.index[-1], color='#808080', alpha=0.1)
ax.set_title(f'Walk-Forward Prediction (24h) vs Reality - {label}', fontsize=16)
ax.set_xlabel('Time Index')
ax.set_ylabel('Number of Rentals')
ax.legend(loc='upper left')
ax.set_xlim(train.index[-168], actual_cnt.index[min(168, len(actual_cnt)-1)])
plt.tight_layout()
plt.show()